In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# cài đặt cấu hình cần thiết
!pip install ultralytics
!pip install -U sahi
!pip install tensorflow matplotlib numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.5 MB/s eta 0:00:00


In [5]:
import os
import cv2
import gc
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# from PIL import Image
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [12]:
#variables
# ai chạy thì đổi lại cái link drive (cũng ichang nhưng mà phải coppy tay nó mới chịu)
folderPath = "/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC /LOBBY/MAIN_SOURCES/"
MODEL_NAME = "yolo26n-seg.pt"
DATASET_NAME = "35_special_images_segmentation.v1i.yolov8"

MODELS_PATH = f"{folderPath}MODELS/"
DATAS_PATH = f"{folderPath}DATASETS/"
TEST_IMAGES_PATH = f"{folderPath}TEST_IMAGES/"

# Lưu kết quả mỗi lần train thành folder với cấu trúc tên dataset + tên model cho dễ kiếm
trainedResult_folder = f"{DATASET_NAME} + {MODEL_NAME}"

# đích đến lưu vào drive
destination= f"{folderPath}RESULTS/{trainedResult_folder}/"

#thư mục test
testImage_folder= f"{TEST_IMAGES_PATH}DETECTABLE_TEST_IMAGES/" #reality path


# ----SAHI------
sahi_detection = f"{folderPath}RESULTS/SAHI/{trainedResult_folder}"
sahi_detection_cropped  = f"{folderPath}/RESULTS/SAHI/CROPPED"


In [6]:
#train stage
model = YOLO(os.path.join(MODELS_PATH, MODEL_NAME))
data =f"{os.path.join(DATAS_PATH, DATASET_NAME)}/data.yaml"

#folder chứa kết quả sau khi train nằm trên content tạm
resultFolder = f"/content/runs/segment/yolo26_rice_seg/{trainedResult_folder}/"

model_result = model.train(
            data = data,
            epochs = 80,
            batch = 16,
            imgsz = 640,
            project = 'yolo26_rice_seg',
            name = trainedResult_folder,
            exist_ok = True #ghi đè lên kết quả cũ nếu đã tồn tại
            )


KeyboardInterrupt: 

In [2]:
# test nhanh xem train ổn ko
model_demo = '/content/runs/segment/yolo26_rice_seg/35_special_images_segmentation.v1i.yolov8 + yolo26n-seg.pt/weights/best.pt'
results = YOLO(model_demo).predict(source='https://media.istockphoto.com/id/186786216/photo/unmilled-rice-grains.jpg?s=612x612&w=0&k=20&c=K0nS9TtIWvwHsLw66vLYOyvYGiqZT9_joqPzwjwBFLY=', save=True)

NameError: name 'YOLO' is not defined

In [7]:
# Lưu kết quả huấn luyện (biểu đồ và weights) vào Google Drive
if os.path.exists(resultFolder):
    shutil.copytree(resultFolder, destination, dirs_exist_ok=True)
    print(f"Đã sao lưu kết quả tại: {destination}")
else:
    print("Không tìm thấy thư mục kết quả huấn luyện tạm thời.")

NameError: name 'resultFolder' is not defined

In [10]:
# Tải model đã huấn luyện từ Drive để sẵn sàng dự đoán
bestModel_path = os.path.join(destination, "weights/best.pt")

if os.path.exists(bestModel_path):
    best_model = YOLO(bestModel_path)
    print("Đã tải model thành công.")
else:
    print("Cảnh báo: Không tìm thấy file weights/best.pt trên Drive.")

Đã tải model thành công.


In [11]:
# os.makedirs(sahi_detection, exist_ok=True)
# os.makedirs(sahi_detection_cropped, exist_ok=True)

# images = [f for f in os.listdir(testImage_folder) if f.lower().endswith(('.jpg', '.png'))]

# print("Đang nạp model vào SAHI...")
# detection_model = AutoDetectionModel.from_pretrained(
#     model_type="ultralytics",
#     model_path=bestModel_path,
#     confidence_threshold=0.4,
#     device="cuda:0"
# )

# for img_name in images:
#     image_path = os.path.join(testImage_folder, img_name)
#     base_name = os.path.splitext(img_name)[0]

#     result = get_sliced_prediction(
#         image_path,
#         detection_model,
#         slice_height=512,
#         slice_width=512,
#         overlap_height_ratio=0.2,
#         overlap_width_ratio=0.2,
#         postprocess_match_threshold=0.6
#         )
#     predictions = result.object_prediction_list

#     img_bgr = cv2.imread(image_path)
#     global_mask = np.zeros(img_bgr.shape[:2], dtype=np.uint8)

#     for idx, obj in enumerate(predictions):
#         if obj.mask is None: continue
#         for polygon in obj.mask.segmentation:
#             pts = np.array(polygon, np.int32).reshape((-1, 1, 2))
#             cv2.fillPoly(global_mask, [pts], 255)

#             ys, xs = np.where(global_mask == 255)
#             if len(ys) > 0:
#                 y1, y2, x1, x2 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
#                 crop_img = img_bgr[y1:y2, x1:x2]
#                 rgba = cv2.cvtColor(crop_img, cv2.COLOR_BGR2BGRA)
#                 rgba[:, :, 3] = global_mask[y1:y2, x1:x2]
#                 cv2.imwrite(os.path.join(sahi_detection_cropped, f"{base_name}_{idx:03d}.png"), rgba)

#     overlay = img_bgr.copy()
#     overlay[global_mask == 255] = [0, 255, 0]
#     cv2.imwrite(os.path.join(sahi_detection, f"{base_name}_seg.jpg"), cv2.addWeighted(overlay, 0.4, img_bgr, 0.6, 0))

#     del result, img_bgr; gc.collect()

# print(f"Hoàn tất! Kết quả tại: {sahi_detection}")

Đang nạp model vào SAHI...
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 9 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 slices.
Performing prediction on 4 s

In [13]:
# cải tiến, sài cái này
import os
import cv2
import numpy as np
import gc
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# Giữ nguyên cấu hình đường dẫn và khởi tạo thư mục của file cũ
os.makedirs(sahi_detection, exist_ok=True)
os.makedirs(sahi_detection_cropped, exist_ok=True)

images = [f for f in os.listdir(testImage_folder) if f.lower().endswith(('.jpg', '.png'))]
print("Đang nạp model vào SAHI...")

# Tối ưu hóa: Bỏ tham số model=best_model dư thừa, chỉ dùng bestModel_path
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=bestModel_path,
    confidence_threshold=0.4,
    device="cuda:0"
)

for img_name in images:
    image_path = os.path.join(testImage_folder, img_name)
    base_name = os.path.splitext(img_name)[0]

    # Thực hiện dự đoán cắt lát (SAHI)
    result = get_sliced_prediction(
        image_path,
        detection_model,
        slice_height=256,
        slice_width=256,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        postprocess_match_threshold=0.7
    )
    predictions = result.object_prediction_list

    img_bgr = cv2.imread(image_path)

    # CẢI TIẾN 1: Khởi tạo mask tổng hợp bên trong vòng lặp của từng ảnh
    global_mask = np.zeros(img_bgr.shape[:2], dtype=np.uint8)

    for idx, obj in enumerate(predictions):
        if obj.mask is None:
            continue

        # CẢI TIẾN 2: Tạo local_mask riêng biệt cho từng hạt để cô lập tọa độ crop
        local_mask = np.zeros(img_bgr.shape[:2], dtype=np.uint8)

        # Duyệt qua toàn bộ polygon của đối tượng (đảm bảo không sót mảnh nào)
        for polygon in obj.mask.segmentation:
            pts = np.array(polygon, np.int32).reshape((-1, 1, 2))
            cv2.fillPoly(local_mask, [pts], 255)       # Vẽ vào mask riêng để crop
            cv2.fillPoly(global_mask, [pts], 255)      # Vẽ vào mask chung để làm overlay

        # Tìm tọa độ biên dựa trên local_mask (Không bị phình to hay dính hạt khác)
        ys, xs = np.where(local_mask == 255)
        if len(ys) > 0 and len(xs) > 0:
            y1, y2 = ys.min(), ys.max() + 1
            x1, x2 = xs.min(), xs.max() + 1

            # Thực hiện cắt ảnh hạt lúa
            crop_img = img_bgr[y1:y2, x1:x2]
            crop_mask = local_mask[y1:y2, x1:x2]

            # Chuyển sang định dạng RGBA để lấy nền trong suốt (Transparent)
            rgba = cv2.cvtColor(crop_img, cv2.COLOR_BGR2BGRA)
            rgba[:, :, 3] = crop_mask

            # Lưu ảnh hạt lúa đã cắt tách biệt hoàn toàn
            cv2.imwrite(os.path.join(sahi_detection_cropped, f"{base_name}_{idx:03d}.png"), rgba)

    # CẢI TIẾN 3: Tạo hiệu ứng vẽ đè màu xanh dạng bán trong suốt (Semi-transparent overlay)
    color_overlay = np.zeros_like(img_bgr, dtype=np.uint8)
    color_overlay[global_mask == 255] = [0, 255, 0] # Đổ màu xanh lên các vùng có hạt

    # Hòa trộn overlay với ảnh gốc theo tỷ lệ alpha (Giữ lại cấu hình bề mặt hạt)
    img_with_masks = cv2.addWeighted(color_overlay, 0.4, img_bgr, 1.0, 0)

    # Lưu ảnh kết quả overlay tổng thể
    cv2.imwrite(os.path.join(sahi_detection, f"{base_name}_seg.jpg"), img_with_masks)

    # CẢI TIẾN 4: Giải phóng triệt để bộ nhớ sau mỗi ảnh để tránh tràn RAM/VRAM
    del result, predictions, img_bgr, global_mask, local_mask, color_overlay, img_with_masks
    gc.collect()

print(f"Hoàn tất! Kết quả tại: {sahi_detection}")

Đang nạp model vào SAHI...
Performing prediction on 1 slices.
Performing prediction on 20 slices.
Performing prediction on 30 slices.
Performing prediction on 30 slices.
Performing prediction on 12 slices.
Hoàn tất! Kết quả tại: /content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC /LOBBY/MAIN_SOURCES/RESULTS/SAHI/35_special_images_segmentation.v1i.yolov8 + yolo26n-seg.pt
